# 데이터 불러오기 및 임포트
  * X에서 직접 수집한 데이터 2,431건 사용 (비마약: 1960건, 마약: 469) 

In [28]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
import torch.nn as nn
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [4]:
df = pd.read_excel("X_drugs_dataset.xlsx")
df.head()

,id,nickname,user_id,text,hashtags,label,Unnamed: 6,Unnamed: 7
0,1,"작대기,처음술,사끼,케이,텔문의 @bing824",@tagar12782,아이스판매 텔(bing824)\n\n아이스파는곳 빙두맛집\n\n케이삽니다 강남아이스...,#찬술|#술친|#술맛집,1.0,NaN,NaN
1,2,비나,@domainVINA,비 오는 날...\n우비 씌운 짱냥이랑 우산 같이 쓰고 쭈그려 앉아서 마당에다 그림...,NaN,0.0,NaN,
2,3,"작대기,얼음술,사끼,빙두,텔문의 @bing824",@JSavasha90744,서울아이스 판매 시원한술 얼음 \n떨액 브액 캔디 케이\n아이스술 차가운술 작대기 ...,NaN,1.0,NaN,NaN
3,4,작대기,@jagdaeg2,작대기 : 범석이는 왜 저럴까,NaN,0.0,NaN,NaN
4,5,고냑이,@muryaong,왴ㅋㅋㅋ 쌍방형인데 언제 작대기 가져감,NaN,0.0,NaN,NaN


# 데이터 정제
  * 줄바꿈 제거, 이모지 제거, 유니코드 제거 등등

In [7]:
pd.set_option('display.max_colwidth', None)

import re

def clean_text(text):

    text = str(text)

    # 영어 소문자
    text = text.lower()

    # 줄바꿈 제거
    text = text.replace('\n', ' ')


    # 이모지 제거
    text = re.sub(
        r'[\U00010000-\U0010ffff]',
        ' ',
        text
    )

    # 특수 유니코드 문자 제거
    text = re.sub(
        r'[⫬·•‧˚｡⋆❅❀꒰໒✿♡♥◟◞]+',
        ' ',
        text
    )

    # 장식용 괄호/따옴표 제거
    text = re.sub(
        r'[\[\]\'"“”‘’(){}]+',
        ' ',
        text
    )

    # 장식용 특수문자 제거
    text = re.sub(
        r'[&*%=+<>]',
        ' ',
        text
    )

    # 감정 표현 혼합 제거
    text = re.sub(
        r'[ㅠㅜㅋㅎ큐]{3,}',
        ' ',
        text
    )

    # 과한 점/슬래시 제거
    text = re.sub(r'[./]{2,}', ' ', text)

    # .. !!! ??? 같은 반복 제거
    text = re.sub(r'([!?.,])\1{1,}', r'\1', text)

    # 의미 없는 특수문자 제거
    text = re.sub(r'[~^`|:;,]', ' ', text)

    # 공백 정리
    text = re.sub(r'\s+', ' ', text).strip()

    # 감정 표현 제거
    text = re.sub(r'[ㅠㅜㅋㅎ큐]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


df = df[["text", "label"]].dropna()
df["text"] = df["text"].astype(str)
df["label"] = df["label"].astype(int)


df['clean_text'] = df['text'].apply(clean_text)
print(df['clean_text'])
selected_df = df[['clean_text', 'label']].copy()



0           아이스판매 텔 bing824 아이스파는곳 빙두맛집 케이삽니다 강남아이스 브액팔아요 도리도리 히로뽕 작대기 브액 찬술 차가운술 시원한술 캔디 케이 오방 ㄹㅅㄱ 캔디 허브 떨 대마초 판매 #찬술 #술친 #술맛집 텔문의 @bing824 http t.me/bing824
1                 비 오는 날 우비 씌운 짱냥이랑 우산 같이 쓰고 쭈그려 앉아서 마당에다 그림 그리는 딩초기려 이거 창호 고양이 그림 뭬? 못그렸어? 웅 나 그려 봐 발톱으로 그릴 수 있어? 움 동그라미 작대기 다섯 개 그림 그 창호야 내가 더 잘 그리는 거 같아 며?
2       서울아이스 판매 시원한술 얼음 떨액 브액 캔디 케이 아이스술 차가운술 작대기 서울 떨 판매 전국좌표 크리스탈 부산아이스 광주아이스 최고 퀄리티 저렴하게 판매 찬술은 코코 오방 ㄹㅅㄱ 텔문의 @bing824 http t.me/bing824 눌러주세요 칼좌표 전국 드랍완료
3                                                                                                                                               작대기 범석이는 왜 저럴까
4                                                                                                                                           왴 쌍방형인데 언제 작대기 가져감
                                                                                 ...                                                                          
2432                                          

# Data Split 및 Class Weight 
  * Train 샘플: 1943건 (비마약: 1568/ 마약: 375)
  * Test 샘플: 486건 (비마약: 392/ 마약: 94)
  * 3-Fold

In [ ]:
train_df, test_df = train_test_split(
    selected_df,
    test_size=0.2,
    random_state=42,
    stratify=selected_df["label"]
)

In [36]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float
)

print("Class Weights")
print(class_weights)

Class Weights
tensor([0.6196, 2.5907])


# Model

In [ ]:
# 기초 설정
#토크나이저 불러오기
MODEL_NAME = "beomi/KcELECTRA-base-v2022"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch['clean_text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

def make_dataset(df):
    dataset = Dataset.from_pandas(df.reset_index(drop=True))
    dataset = dataset.map(tokenize, batched=True)
    dataset = dataset.rename_column("label", "labels")
    dataset = dataset.remove_columns(["clean_text"])
    dataset.set_format("torch")
    return dataset
train_dataset = make_dataset(train_df)
test_dataset = make_dataset(test_df)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = class_weights.to(logits.device)
        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss
    
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    preds = np.argmax(logits, axis=1)
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
        "roc_auc": roc_auc_score(labels, probs)
    }

Map: 100%|██████████| 486/486 [00:00<00:00, 19424.37 examples/s]


In [39]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

training_args = TrainingArguments(
    output_dir="./kcelectra_drug_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none"
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 613420.85it/s]
[transformers] ElectraForSequenceClassification LOAD REPORT from: beomi/KcELECTRA-base-v2022
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because 

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,No log,0.361318,0.944444,0.894118,0.808511,0.849162,0.952698
2,No log,0.379902,0.940329,0.865169,0.819149,0.841530,0.953919


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.37it/s]


TrainOutput(global_step=486, training_loss=0.30308520156169627, metrics={'train_runtime': 110.9137, 'train_samples_per_second': 35.036, 'train_steps_per_second': 4.382, 'total_flos': 255612390282240.0, 'train_loss': 0.30308520156169627, 'epoch': 2.0})

# Evaluation
  * Accuracy, Precision, Recall, F1-Score, ROC-AUC

In [ ]:

eval_result = trainer.evaluate()
print(eval_result)

trainer.save_model("./kcelectra_drug_detection")
tokenizer.save_pretrained("./kcelectra_drug_detection")

print("Model saved.")

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Roc Auc
No log,0.361318,2,0.944444,0.894118,0.808511,0.849162,0.952698


{'eval_loss': 0.36131751537323, 'eval_accuracy': 0.9444444444444444, 'eval_precision': 0.8941176470588236, 'eval_recall': 0.8085106382978723, 'eval_f1': 0.8491620111731844, 'eval_roc_auc': 0.9526975683890577}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.19it/s]

Model saved.
